# FarmTech Solutions – Fase 6: Visão Computacional com YOLO

**Grupo:**
- Jonattas Felipe – RM572692
- Natanael Filho – RM572474
- Pedro Henrique – RM571394
- Andrews Oliveira – RM572311

Notebook organizado em:
1. Preparação do dataset (FiftyOne / Open Images) ← *esta seção está pronta*
2. Rotulação no Make Sense
3. Entrega 1: YOLOv5 customizado (30 e 60 épocas)
4. Entrega 2: YOLO padrão e CNN do zero, com comparação final
5. Conclusões

**Objetos escolhidos:** Trator (A) e Vaca/gado (B), alinhados ao contexto agro da FarmTech.

## 1. Preparação do dataset

Para montar o banco de imagens usei a biblioteca **FiftyOne**, que baixa imagens do dataset público **Open Images V7**. Baixo apenas as **imagens**: a rotulação das caixas será feita por mim no Make Sense (etapa 2).

Passos: baixar candidatas de cada classe, manter só imagens com **um único objeto da classe** (facilita a rotulação e evita ambiguidade), sortear 40 por classe e dividir em **32 treino / 4 validação / 4 teste**.

In [ ]:
# Instala a FiftyOne (pode levar 1-2 minutos)
!pip install -q fiftyone

In [ ]:
# Conecta ao Google Drive para guardar o dataset
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/FarmTech_Fase6'   # pasta do projeto no Drive
!mkdir -p {BASE}

In [ ]:
import random, shutil
from pathlib import Path
from PIL import Image
import fiftyone.zoo as foz

random.seed(42)  # reprodutibilidade do sorteio

# Classe do Open Images -> nome usado no projeto
CLASSES = {'Tractor': 'trator', 'Cattle': 'vaca'}
N_POR_CLASSE = 40
SPLITS = {'train': 32, 'val': 4, 'test': 4}

# Estrutura de pastas no formato esperado pelo YOLOv5
DATASET = Path(BASE) / 'dataset'
for s in SPLITS:
    (DATASET / 'images' / s).mkdir(parents=True, exist_ok=True)
    (DATASET / 'labels' / s).mkdir(parents=True, exist_ok=True)

In [ ]:
def baixar_e_filtrar(classe_oi, nome):
    """Baixa candidatas do Open Images e mantém só as que têm exatamente
    1 objeto da classe e nenhuma das outras classes do projeto."""
    ds = foz.load_zoo_dataset(
        'open-images-v7', split='train',
        label_types=['detections'], classes=[classe_oi],
        max_samples=250, only_matching=True,
        dataset_name=f'oi_{nome}', drop_existing_dataset=True,
    )
    outras = set(CLASSES) - {classe_oi}
    aprovadas = []
    for amostra in ds:
        rotulos = [d.label for d in amostra.ground_truth.detections]
        if rotulos.count(classe_oi) == 1 and not (set(rotulos) & outras):
            aprovadas.append(amostra.filepath)
    print(f'{nome}: {len(aprovadas)} imagens candidatas após o filtro')
    return aprovadas

candidatas = {nome: baixar_e_filtrar(c, nome) for c, nome in CLASSES.items()}

In [ ]:
def salvar(caminho_origem, destino, lado_max=640):
    """Redimensiona (lado maior = 640 px) e salva em JPG."""
    img = Image.open(caminho_origem).convert('RGB')
    img.thumbnail((lado_max, lado_max))
    img.save(destino, 'JPEG', quality=92)

contagem = {}
for nome, caminhos in candidatas.items():
    assert len(caminhos) >= N_POR_CLASSE, f'Poucas imagens de {nome}; aumente max_samples'
    escolhidas = random.sample(caminhos, N_POR_CLASSE)
    i = 0
    for split, qtd in SPLITS.items():
        for caminho in escolhidas[i:i + qtd]:
            i += 1
            salvar(caminho, DATASET / 'images' / split / f'{nome}_{i:02d}.jpg')
        contagem[(nome, split)] = qtd

for s in SPLITS:
    print(s, len(list((DATASET / 'images' / s).glob('*.jpg'))), 'imagens')

In [ ]:
# Confere visualmente uma amostra de cada classe
import matplotlib.pyplot as plt
fig, eixos = plt.subplots(2, 4, figsize=(14, 7))
for linha, nome in enumerate(CLASSES.values()):
    fotos = sorted((DATASET / 'images' / 'train').glob(f'{nome}_*.jpg'))[:4]
    for ax, f in zip(eixos[linha], fotos):
        ax.imshow(Image.open(f)); ax.set_title(f.name); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Gera um ZIP das imagens para baixar e rotular no Make Sense
shutil.make_archive(f'{BASE}/imagens_para_rotular', 'zip', DATASET / 'images')
print('ZIP salvo em:', f'{BASE}/imagens_para_rotular.zip')

**Fonte das imagens:** Open Images V7 (Google), acessado via FiftyOne. Imagens usadas apenas para fins acadêmicos.

## 2. Rotulação no Make Sense

1. Acessar https://www.makesense.ai, clicar em *Get Started* e carregar as imagens de `images/train` (depois repetir o processo para `val` e `test`).
2. Escolher *Object Detection* e criar as labels **nesta ordem**: `trator` (id 0) e `vaca` (id 1).
3. Desenhar uma caixa ao redor do objeto em cada imagem e associar a label.
4. Em *Actions > Export Annotations*, escolher **A .zip package containing files in YOLO format**.
5. Salvar os três ZIPs no Drive com estes nomes: `labels_zips/train.zip`, `labels_zips/val.zip` e `labels_zips/test.zip`.

A célula abaixo extrai os `.txt` para a pasta `labels/` correta e confere se todas as imagens têm rótulo.

In [ ]:
# Extrai os rótulos exportados do Make Sense para labels/train, val e test
for s in SPLITS:
    zip_path = f'{BASE}/labels_zips/{s}.zip'
    destino = DATASET / 'labels' / s
    !unzip -o -j -q "{zip_path}" -d "{destino}"

# Confere: toda imagem precisa ter um .txt de mesmo nome, com classe 0 ou 1
for s in SPLITS:
    imgs = sorted((DATASET / 'images' / s).glob('*.jpg'))
    sem_label = [i.name for i in imgs if not (DATASET / 'labels' / s / (i.stem + '.txt')).exists()]
    classes_usadas = set()
    for t in (DATASET / 'labels' / s).glob('*.txt'):
        for linha in t.read_text().splitlines():
            if linha.strip():
                classes_usadas.add(int(linha.split()[0]))
    print(f'{s}: {len(imgs)} imagens | sem rótulo: {sem_label} | classes: {classes_usadas}')

## 3. Entrega 1 – YOLOv5 customizado

**Passo a passo:**
1. **Preparo:** copiar o dataset para o disco local do Colab (o treino fica bem mais rápido que lendo do Drive) e criar o `data.yaml`.
2. **Treino:** *fine-tuning* do `yolov5s.pt` (pré-treinado no COCO) com nossas duas classes, em duas simulações: **30 e 60 épocas**.
3. **Validação:** curvas de erro (loss), precisão, recall e mAP calculadas pelo YOLO a cada época sobre a pasta `val`.
4. **Teste:** avaliação do `best.pt` nas imagens da pasta `test`, nunca vistas no treino, e visualização das detecções.

In [ ]:
import time, shutil, pandas as pd, numpy as np
import matplotlib.pyplot as plt

# Copia o dataset para o disco local do Colab (treino mais rápido)
LOCAL = Path('/content/dataset_local')
shutil.copytree(DATASET, LOCAL, dirs_exist_ok=True)

# Arquivo de configuração do dataset para o YOLOv5
yaml_texto = '''path: /content/dataset_local
train: images/train
val: images/val
test: images/test
nc: 2
names: ['trator', 'vaca']
'''
Path('/content/data.yaml').write_text(yaml_texto)
shutil.copy('/content/data.yaml', f'{BASE}/data.yaml')

# Clona o YOLOv5 e instala dependências
!git clone -q https://github.com/ultralytics/yolov5 /content/yolov5
%cd /content/yolov5
!pip install -q -r requirements.txt

In [ ]:
# Simulação 1: 30 épocas
t0 = time.time()
!python train.py --img 640 --batch 8 --epochs 30 --data /content/data.yaml --weights yolov5s.pt --name exp_30ep --exist-ok
tempo_treino_30 = time.time() - t0
print(f'Tempo de treino (30 épocas): {tempo_treino_30/60:.1f} min')

In [ ]:
# Simulação 2: 60 épocas
t0 = time.time()
!python train.py --img 640 --batch 8 --epochs 60 --data /content/data.yaml --weights yolov5s.pt --name exp_60ep --exist-ok
tempo_treino_60 = time.time() - t0
print(f'Tempo de treino (60 épocas): {tempo_treino_60/60:.1f} min')

### 3.1 Validação: curvas de treino

O YOLO registra em `results.csv` as métricas de cada época. Abaixo comparo as duas simulações.

In [ ]:
def ler_resultados(nome):
    df = pd.read_csv(f'/content/yolov5/runs/train/{nome}/results.csv')
    df.columns = df.columns.str.strip()
    return df

res = {'30 épocas': ler_resultados('exp_30ep'), '60 épocas': ler_resultados('exp_60ep')}

# Métricas da última época de cada simulação (validação)
linhas = []
for nome, df in res.items():
    u = df.iloc[-1]
    linhas.append({'Simulação': nome,
                   'Precisão': u['metrics/precision'], 'Recall': u['metrics/recall'],
                   'mAP@0.5': u['metrics/mAP_0.5'], 'mAP@0.5:0.95': u['metrics/mAP_0.5:0.95'],
                   'Loss val (box)': u['val/box_loss']})
tabela_val = pd.DataFrame(linhas).round(3)
display(tabela_val)

# Gráficos comparativos
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for nome, df in res.items():
    ax[0].plot(df['train/box_loss'], label=nome)
    ax[1].plot(df['val/box_loss'], label=nome)
    ax[2].plot(df['metrics/mAP_0.5'], label=nome)
for a, t in zip(ax, ['Erro de treino (box loss)', 'Erro de validação (box loss)', 'mAP@0.5 na validação']):
    a.set_title(t); a.set_xlabel('época'); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

### 3.2 Teste: métricas e imagens processadas

Avaliação do `best.pt` de cada simulação nas 8 imagens da pasta `test`.

In [ ]:
resultados_teste = {}
for nome, exp in [('30 épocas', 'exp_30ep'), ('60 épocas', 'exp_60ep')]:
    saida = !python val.py --weights runs/train/{exp}/weights/best.pt --data /content/data.yaml --task test --img 640 --name test_{exp} --exist-ok 2>&1
    linha = [l for l in saida if l.strip().startswith('all')][0].split()
    P, R, m50, m95 = map(float, linha[3:7])
    resultados_teste[nome] = {'Precisão': P, 'Recall': R, 'mAP@0.5': m50, 'mAP@0.5:0.95': m95}
    print(nome, resultados_teste[nome])

display(pd.DataFrame(resultados_teste).T.round(3))

In [ ]:
# Detecção nas imagens de teste (resultados em runs/detect/detect_XXep)
for exp in ['exp_30ep', 'exp_60ep']:
    !python detect.py --weights runs/train/{exp}/weights/best.pt --source /content/dataset_local/images/test --img 640 --conf-thres 0.25 --name detect_{exp} --exist-ok

# Mostra as imagens processadas pelo modelo de 60 épocas
fotos = sorted(Path('/content/yolov5/runs/detect/detect_exp_60ep').glob('*.jpg'))
fig, eixos = plt.subplots(2, 4, figsize=(18, 9))
for ax, f in zip(eixos.ravel(), fotos):
    ax.imshow(Image.open(f)); ax.set_title(f.name); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Mostra também o modelo de 30 épocas, para comparar visualmente
fotos = sorted(Path('/content/yolov5/runs/detect/detect_exp_30ep').glob('*.jpg'))
fig, eixos = plt.subplots(2, 4, figsize=(18, 9))
for ax, f in zip(eixos.ravel(), fotos):
    ax.imshow(Image.open(f)); ax.set_title(f.name); ax.axis('off')
plt.tight_layout(); plt.show()

# Salva todos os resultados no Drive
shutil.copytree('/content/yolov5/runs', f'{BASE}/runs', dirs_exist_ok=True)
print('Resultados salvos em', f'{BASE}/runs')

## 4. Entrega 2 – Comparação de abordagens

Comparo três abordagens sobre a **mesma base de teste**:

| Abordagem | Descrição |
|---|---|
| **A. YOLO customizado** | YOLOv5s com fine-tuning nas nossas classes (Entrega 1) |
| **B. YOLO padrão** | YOLOv5s com pesos originais do COCO, **sem treinar**. O COCO tem a classe `cow`, mas **não tem `trator`**; usei `truck` como equivalente aproximado |
| **C. CNN do zero** | Rede convolucional pequena treinada só com nossas 64 imagens, classificando a imagem inteira |

**Métrica comum:** acurácia por imagem. Nos YOLOs, a classe da detecção com maior confiança define a classe da imagem; sem detecção conta como erro. Também meço o **tempo médio de inferência** por imagem.

In [ ]:
import torch

pasta_teste = LOCAL / 'images' / 'test'
arquivos_teste = sorted(pasta_teste.glob('*.jpg'))

def avaliar_yolo(modelo, mapa=None):
    modelo.conf = 0.25
    modelo(str(arquivos_teste[0]))                       # aquecimento (não entra na medida)
    acertos, tempos, previsoes = 0, [], []
    for f in arquivos_teste:
        verdade = f.name.split('_')[0]
        t0 = time.perf_counter()
        r = modelo(str(f), size=640)
        tempos.append(time.perf_counter() - t0)
        df = r.pandas().xyxy[0]
        if mapa is not None:                             # traduz classes do COCO para as nossas
            df = df[df['name'].isin(mapa)].copy()
            df['name'] = df['name'].map(mapa)
        pred = df.sort_values('confidence', ascending=False)['name'].iloc[0] if len(df) else 'nenhum'
        previsoes.append((f.name, verdade, pred))
        acertos += (pred == verdade)
    return acertos / len(arquivos_teste), np.mean(tempos) * 1000, previsoes

# A. YOLO customizado (melhor modelo: 60 épocas; troque para 30 se preferir)
yolo_custom = torch.hub.load('/content/yolov5', 'custom', path='/content/yolov5/runs/train/exp_60ep/weights/best.pt', source='local')
acc_A, ms_A, prev_A = avaliar_yolo(yolo_custom)

# B. YOLO padrão (COCO): cow -> vaca, truck -> trator (aproximação)
yolo_padrao = torch.hub.load('/content/yolov5', 'yolov5s', source='local')
acc_B, ms_B, prev_B = avaliar_yolo(yolo_padrao, mapa={'cow': 'vaca', 'truck': 'trator'})

print(f'A) YOLO customizado: acurácia {acc_A:.0%} | {ms_A:.1f} ms/imagem')
print(f'B) YOLO padrão:      acurácia {acc_B:.0%} | {ms_B:.1f} ms/imagem')
print('\nPrevisões do YOLO padrão (arquivo, verdade, previsto):')
for p in prev_B: print(p)

### 4.1 CNN treinada do zero

Rede pequena (3 blocos convolucionais) com *data augmentation* (espelhamento, rotação e zoom), porque 64 imagens de treino é muito pouco. Ela só diz **qual classe** a imagem tem; não localiza o objeto.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

LADO = 128

def carregar(split):
    X, y = [], []
    for f in sorted((LOCAL / 'images' / split).glob('*.jpg')):
        X.append(np.array(Image.open(f).convert('RGB').resize((LADO, LADO))))
        y.append(1 if f.name.startswith('vaca') else 0)     # 0 = trator, 1 = vaca
    return np.array(X, dtype='float32'), np.array(y)

Xtr, ytr = carregar('train'); Xva, yva = carregar('val'); Xte, yte = carregar('test')
print(Xtr.shape, Xva.shape, Xte.shape)

cnn = models.Sequential([
    layers.Input((LADO, LADO, 3)),
    layers.Rescaling(1/255),                                 # normaliza pixels para 0-1
    layers.RandomFlip('horizontal'), layers.RandomRotation(0.1), layers.RandomZoom(0.15),
    layers.Conv2D(16, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'), layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid'),                   # probabilidade de ser vaca
])
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.summary()

In [ ]:
parar = callbacks.EarlyStopping(patience=12, restore_best_weights=True, monitor='val_loss')

t0 = time.time()
hist = cnn.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=60, batch_size=16, callbacks=[parar], verbose=2)
tempo_treino_cnn = time.time() - t0
print(f'Tempo de treino da CNN: {tempo_treino_cnn:.1f} s')

# Curvas de aprendizado
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist.history['loss'], label='treino'); ax[0].plot(hist.history['val_loss'], label='validação'); ax[0].set_title('Erro (loss)')
ax[1].plot(hist.history['accuracy'], label='treino'); ax[1].plot(hist.history['val_accuracy'], label='validação'); ax[1].set_title('Acurácia')
for a in ax: a.legend(); a.grid(alpha=.3); a.set_xlabel('época')
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

pred_cnn = (cnn.predict(Xte, verbose=0).ravel() > 0.5).astype(int)
acc_C = (pred_cnn == yte).mean()

# Tempo de inferência por imagem (uma a uma, igual aos YOLOs)
cnn.predict(Xte[:1], verbose=0)                              # aquecimento
tempos = []
for i in range(len(Xte)):
    t0 = time.perf_counter(); cnn.predict(Xte[i:i+1], verbose=0); tempos.append(time.perf_counter() - t0)
ms_C = np.mean(tempos) * 1000

print(f'C) CNN do zero: acurácia {acc_C:.0%} | {ms_C:.1f} ms/imagem')
ConfusionMatrixDisplay(confusion_matrix(yte, pred_cnn), display_labels=['trator', 'vaca']).plot()
plt.title('CNN do zero – matriz de confusão (teste)'); plt.show()

### 4.2 Comparação final

In [ ]:
comparacao = pd.DataFrame({
    'Abordagem': ['A. YOLO customizado (60 ép.)', 'B. YOLO padrão (COCO)', 'C. CNN do zero'],
    'Acurácia no teste': [acc_A, acc_B, acc_C],
    'Tempo de treino (min)': [tempo_treino_60 / 60, 0.0, tempo_treino_cnn / 60],
    'Inferência (ms/imagem)': [ms_A, ms_B, ms_C],
    'Localiza o objeto?': ['Sim', 'Sim', 'Não'],
}).round(3)
display(comparacao)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
nomes = ['A. YOLO\ncustom', 'B. YOLO\npadrão', 'C. CNN\ndo zero']
ax[0].bar(nomes, comparacao['Acurácia no teste']); ax[0].set_title('Acurácia no teste'); ax[0].set_ylim(0, 1.05)
ax[1].bar(nomes, comparacao['Tempo de treino (min)']); ax[1].set_title('Tempo de treino (min)')
ax[2].bar(nomes, comparacao['Inferência (ms/imagem)']); ax[2].set_title('Inferência (ms/imagem)')
plt.tight_layout(); plt.show()

**Comparação crítica** *(complete com os seus números depois de executar)*

| Critério | A. YOLO customizado | B. YOLO padrão | C. CNN do zero |
|---|---|---|---|
| **Facilidade de uso/integração** | Média: exige rotular caixas, treinar e gerenciar o repositório YOLOv5, mas depois entrega `best.pt` pronto para usar (inclusive em webcam/ESP32-CAM). | Alta: zero treino, funciona em poucas linhas. | Média: código curto em Keras, mas é preciso projetar a arquitetura e ajustar hiperparâmetros. |
| **Precisão** | *(preencher)* | *(preencher: cobre `vaca`, mas `trator` não existe no COCO)* | *(preencher)* |
| **Tempo de treino** | *(preencher)* | Nenhum, pesos já prontos. | *(preencher)* |
| **Tempo de inferência** | *(preencher)* | *(preencher)* | *(preencher)* |

Pontos que vale discutir no texto:
- O YOLO padrão não conhece trator, então tende a errar essa classe. Isso mostra por que o *fine-tuning* é necessário quando o objeto não está no COCO.
- A CNN só classifica a imagem inteira, não diz **onde** está o objeto. Serve se cada foto tem um único objeto; o YOLO resolve cenas com vários.
- Com apenas 8 imagens de teste, cada erro pesa 12,5 pontos percentuais. Os números são indicativos, não conclusivos.

## 5. Conclusões

*(Escreva com suas palavras, apoiado nas tabelas e gráficos acima.)*

**Efeito do número de épocas (30 × 60):** compare o mAP@0.5, o erro de validação e o tempo de treino. Perguntas para guiar o texto: o modelo de 60 épocas melhorou de forma relevante? Houve sinal de overfitting (erro de treino caindo e o de validação subindo)? O custo extra de tempo compensou?

**Resultados no teste:** o que as imagens processadas mostram? Há detecções com caixa mal posicionada, confiança baixa ou objetos não detectados? Cole aqui os *prints* mais convincentes para o cliente FarmTech.

**Pontos fortes:** detecção e localização com poucas imagens graças ao *fine-tuning*; modelo leve (`yolov5s`); `best.pt` reutilizável em tempo real.

**Limitações:**
- Dataset pequeno (80 imagens) e conjunto de teste de só 8 imagens, o que torna as métricas instáveis.
- Imagens do Open Images têm cenários variados, mas a base foi filtrada para um objeto por foto, então o modelo não foi testado em cenas com vários objetos.
- Uso de `truck` como substituto de `trator` na comparação com o YOLO padrão é uma aproximação.

**Próximos passos:** mais imagens, *transfer learning* com MobileNet/VGG (Ir Além), e testar o `best.pt` em tempo real com webcam ou ESP32-CAM.